# Можно ли сверить остаток по накладным?

Учебный склад начал день с 30 упаковками. В конце дня вам передали CSV из учётной системы. Подготовьте отчёт о движении упаковок и сохраните номер строки, которую придётся уточнить у кладовщика.

Первая строка содержит имена полей. Разделитель — точка с запятой. Значение qty записано положительным целым числом. Направление задаёт отдельное поле kind со значением «приход» или «выдача». Строку с нарушением этих правил пропустите и укажите причину. Текст файла в упражнении открывается через StringIO.

Выполняйте ячейки по порядку. Перед запуском попробуйте предсказать вывод.

## Шаг 1. Как прочитать накладные, если в названии товара встречается сам разделитель?

In [ ]:
from io import StringIO
import csv

text = """kind;item;qty
выдача;"Чай; зелёный";3
приход;Кофе;8
выдача;Кофе;два
выдача;Чай;4
"""
rows = list(csv.DictReader(StringIO(text), delimiter=";"))
print(len(rows))
print(rows[0]["item"], rows[0]["qty"], sep=" | ")

**Обсуждение.** DictReader берёт имена полей из заголовка и сохраняет каждую накладную как словарь. Кавычки защищают точку с запятой внутри названия, поэтому всё название остаётся в одном поле. Количество на этом этапе всё ещё строка, даже когда его печатное представление похоже на число.

## Шаг 2. Как отделить пригодные операции и оставить след для исправления источника?

In [ ]:
events = []
issues = []
for line, row in enumerate(rows, start=2):
    try:
        qty = int(row["qty"])
    except ValueError:
        issues.append((line, "количество не целое"))
        continue
    if qty <= 0 or row["kind"] not in ("приход", "выдача"):
        issues.append((line, "проверьте вид операции и количество"))
        continue
    events.append((row["kind"], qty))
print(events)
print(issues)

**Обсуждение.** В этой выгрузке каждая запись занимает одну физическую строку. Отсчёт начинается с двух из-за заголовка. До вычислений доходят три операции, а четвёртая строка файла с текстом «два» получает отметку для проверки.

## Шаг 3. Как сверить остаток двумя суммами и назвать пределы этого расчёта?

In [ ]:
incoming = sum(qty for kind, qty in events if kind == "приход")
outgoing = sum(qty for kind, qty in events if kind == "выдача")
balance = 30 + incoming - outgoing
print("Приход:", incoming, "Выдача:", outgoing)
print("Остаток по принятым строкам:", balance)
print("Принято:", len(events), "Требует проверки:", len(issues))

**Обсуждение.** Оприходованы восемь упаковок, выданы три и четыре. С учётом начальных тридцати расчёт даёт 31 упаковку. Пока одна строка ждёт уточнения, это предварительный остаток. Отчёт показывает, откуда берётся неопределённость.

## Поиск логической ошибки
Почему результат может ввести в заблуждение?

In [ ]:
initial = 30
sample = [("выдача", 3), ("приход", 8), ("выдача", 4)]
print(initial + sum(qty for kind, qty in sample))

Каждое количество в файле записано положительным числом, даже когда упаковки покидают склад. Если просто сложить все три значения, выдача тоже увеличит остаток. Сначала отделите приход от выдачи, затем вычтите суммарную выдачу.

## Измените задачу
Кладовщик подтвердил, что «два» в четвёртой строке означает выдачу двух упаковок кофе. Замените значение в исходном тексте на 2 и выполните все ячейки заново. Как поменяются остаток и число принятых записей?

In [ ]:
# Ваше решение

Для проверки: Приход останется 8, выдача станет 9. Итог равен 30 + 8 − 9 = 29 упаковкам. Теперь приняты все четыре записи, строк для уточнения нет.